# Exercise 2 — validate_ohlcv

Before trusting any market data, validate it. `validate_ohlcv` checks four things: it's a DataFrame, it has all five OHLCV columns, it's not empty, and High is never below Low. It returns `(True, "")` on success or `(False, reason)` on failure — never raises.

In [ ]:
import pandas as pd, math, sqlite3, tempfile, os

OHLCV_COLS = ["Open", "High", "Low", "Close", "Volume"]

def _synthetic(ticker="TEST", period="1y", interval="1d", n=50):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })

# ── Exercise: implement validate_ohlcv ───────────────────────────────────────

def validate_ohlcv(df):
    """Validate that df is a well-formed OHLCV DataFrame.

    Checks (in order):
      1. df is a pd.DataFrame
      2. All five OHLCV_COLS are present
      3. df has at least one row
      4. High >= Low for every non-NaN row

    Returns:
        (True, "")  if all checks pass
        (False, reason_string)  on first failure
    Never raises.
    """
    # TODO:
    # 1. if not isinstance(df, pd.DataFrame): return False, "not a DataFrame"
    # 2. find missing = [c for c in OHLCV_COLS if c not in df.columns]
    #    if missing: return False, "missing columns: " + ", ".join(missing)
    # 3. if len(df) == 0: return False, "DataFrame is empty"
    # 4. valid = df.dropna(subset=["High", "Low"])
    #    if len(valid) > 0 and (valid["High"] < valid["Low"]).any():
    #        return False, "High < Low detected"
    # 5. return True, ""
    return True, ""


### Checks

In [ ]:
checks = 0

# 1 — valid synthetic data returns (True, "")
try:
    df = _synthetic()
    ok, reason = validate_ohlcv(df)
    assert ok, f"expected ok=True, got reason={reason!r}"
    assert reason == ""
    checks += 1; print("✅ 1 valid OHLCV returns (True, '')")
except Exception as e:
    print("❌ 1:", e)

# 2 — missing column returns (False, reason mentioning the column)
try:
    df_bad = _synthetic().drop(columns=["Volume"])
    ok, reason = validate_ohlcv(df_bad)
    assert not ok, "expected ok=False for missing Volume"
    assert "Volume" in reason, f"reason should mention 'Volume': {reason!r}"
    checks += 1; print("✅ 2 missing column returns (False, reason)")
except Exception as e:
    print("❌ 2:", e)

# 3 — empty DataFrame returns (False, reason)
try:
    df_empty = pd.DataFrame(columns=OHLCV_COLS)
    ok, reason = validate_ohlcv(df_empty)
    assert not ok, "expected ok=False for empty DataFrame"
    checks += 1; print("✅ 3 empty DataFrame returns (False, reason)")
except Exception as e:
    print("❌ 3:", e)

# 4 — High < Low returns (False, reason)
try:
    df_inv = _synthetic().copy()
    df_inv.loc[df_inv.index[5], "High"] = df_inv["Low"].iloc[5] - 1.0
    ok, reason = validate_ohlcv(df_inv)
    assert not ok, "expected ok=False when High < Low"
    assert "High" in reason or "Low" in reason or "detected" in reason
    checks += 1; print("✅ 4 High < Low returns (False, reason)")
except Exception as e:
    print("❌ 4:", e)

# 5 — non-DataFrame returns (False, reason); never raises
try:
    for bad in ["string", 42, None, [1, 2, 3]]:
        ok, reason = validate_ohlcv(bad)
        assert not ok, f"expected ok=False for {type(bad).__name__}"
    checks += 1; print("✅ 5 non-DataFrame inputs return (False, reason), never raise")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
